# Generación y Simulación de Datos Financieros

**SIMULACIÓN** > EDA > entrenamiento

Simulación y generación de datos sintéticos de usuarios y transacciones. Su salida alimentará al análisis exploratorio y servirá de semilla para poblar la base de datos local.

## 1. Preparación del Entorno

In [1]:
import pandas as pd
import numpy as np
import os

SEED = 42
np.random.seed(SEED)

## 2. Simulación de 1000 Usuarios

Probando con un sistema de puntuación para definir el perfil financiero de cada usuario:
- **Ingreso mensual**: >3500 (3 pts), 2000-3500 (2 pts), 1000-2000 (1 pt), <1000 (0 pts).
- **Nivel de endeudamiento**: <=20% (4 pts), 21%-39% (2 pts), 40%-60% (1 pt), >60% (0 pts).
- **Frecuencia de ahorro**: Alta (3), Media (2), Baja (1), Ninguna (0).
- **Clasificación final**: Saludable (8-10 pts), En observacion (5-7 pts), En riesgo (0-4 pts).

In [2]:
n_usuarios = 1000

nombres = ['Juan', 'Maria', 'Carlos', 'Ana', 'Luis', 'Lucia', 'Pedro', 'Sofia', 'Diego', 'Laura', 
           'Jose', 'Claudia', 'Jorge', 'Elena', 'Miguel', 'Marta', 'Martin', 'Valentina', 'Alejandro', 'Camila']

# Generación aleatoria de atributos base
ingresos = np.random.randint(500, 6000, size=n_usuarios) # Ingresos entre 500 y 6000
endeudamiento = np.random.randint(0, 90, size=n_usuarios) # Endeudamiento entre 0-90%
# 3 pasos para generar frecuencia de ahorro
ahorro_opciones = ['Alta', 'Media', 'Baja', 'Ninguna'] # Setear
ahorro_prob = [0.2, 0.4, 0.25, 0.15] # Distribuir. Define la distribución de probabilidades definidas arriba para los 1000 usuarios. Esta distribución puede cambiarse, pero tiene sentido que se mantenga similar a esta
ahorro_frecuencia = np.random.choice(ahorro_opciones, size=n_usuarios, p=ahorro_prob) # Generar

# Fórmula scoring financiero (propuesta)
perfiles = []
for i in range(n_usuarios):
    pts = 0
    # Puntos por Ingresos
    if ingresos[i] > 3500:
        pts += 3
    elif ingresos[i] >= 2000:
        pts += 2
    elif ingresos[i] >= 1000:
        pts += 1
        
    # Puntos por Nivel de Endeudamiento
    if endeudamiento[i] <= 20:
        pts += 4
    elif endeudamiento[i] < 40:
        pts += 2
    elif endeudamiento[i] <= 60:
        pts += 1
        
    # Puntos por Frecuencia de Ahorro
    if ahorro_frecuencia[i] == 'Alta':
        pts += 3
    elif ahorro_frecuencia[i] == 'Media':
        pts += 2
    elif ahorro_frecuencia[i] == 'Baja':
        pts += 1
        
    # Perfil Resultante
    #  Tildes = conflictos de encoding  (UTF-8/ISO-8859-1) entre Python, Java REST API y Vue.js
    if pts >= 8:
        perf = 'Saludable'
    elif pts >= 5:
        perf = 'En observacion'
    else:
        perf = 'En riesgo'
    perfiles.append(perf)

# Ensamblaje
df_usuarios = pd.DataFrame({
    'id': range(1, n_usuarios + 1),
    'nombre': np.random.choice(nombres, size=n_usuarios),
    'ingreso_mensual': ingresos,
    'nivel_endeudamiento': endeudamiento,
    'frecuencia_ahorro': ahorro_frecuencia,
    'perfil_financiero': perfiles
})

print("Usuarios simulados:", df_usuarios.shape) # 1000 usuarios, 6 col
print(df_usuarios['perfil_financiero'].value_counts())

Usuarios simulados: (1000, 6)
perfil_financiero
En observacion    476
En riesgo         343
Saludable         181
Name: count, dtype: int64


## 3. Simulación de 5000 transacciones

Para el modelo de procesamiento de lenguaje natural (NLP)

In [3]:
n_transacciones = 5000

# Conceptos por categorías de gastos en minúsculas
conceptos_categoria = {
    'Alimentacion': [
        'supermercado coto', 'verduleria de la esquina', 'carniceria central', 
        'almacen san martin', 'compra panaderia', 'supermercado carrefour', 
        'verduleria y fruteria', 'compras fiambreria', 'compra supermercado dia'
    ],
    'Transporte': [
        'viaje uber', 'colectivo linea 60', 'carga sube', 'combustible ypf', 
        'peaje autopista', 'taxis de la ciudad', 'viaje cabify', 'combustible shell'
    ],
    'Salud': [
        'estudios clinicos', 'compra farmacia', 'consulta medica de control', 
        'cuota prepaga osde', 'medicamentos recetados', 'dentista limpieza', 
        'analisis de sangre', 'optica lentes'
    ],
    'Vivienda': [
        'pago alquiler mensual', 'expensas del edificio', 'servicio de plomeria', 
        'reparacion de cerradura', 'compra ferreteria', 'pintura habitacion', 
        'expensas comunes', 'reparacion electrica'
    ],
    'Educacion': [
        'cuota universidad', 'compra libros de texto', 'curso de programacion', 
        'matricula del colegio', 'utiles escolares', 'taller de ingles', 
        'cuota jardin de infantes'
    ],
    'Ocio': [
        'salida al cine', 'suscripcion netflix', 'cena restaurante', 
        'entradas recital', 'cerveceria artesanal', 'suscripcion spotify', 
        'juegos de consola', 'compra cafeteria'
    ],
    'Servicios': [
        'factura luz edesur', 'abono internet fibertel', 'servicio de agua aysa', 
        'factura gas metrogas', 'abono telefonia movil', 'impuestos municipales'
    ],
    'Deudas': [
        'pago tarjeta de credito', 'cuota prestamo banco', 'pago de intereses', 
        'cuota prestamo auto', 'pago saldo refinanciado'
    ],
    'Ahorros': [
        'transferencia ahorro mensual', 'compra dolares banco', 'plazo fijo', 
        'fondo comun de inversion', 'inversion en acciones'
    ]
}

categorias = list(conceptos_categoria.keys()) # Lista las categorias sólas
usuario_ids = np.random.randint(1, n_usuarios + 1, size=n_transacciones) # Inventa los ids
selected_categories = np.random.choice(categorias, size=n_transacciones) # Array. Asigna aleatoriamente una categoría y una descripción de las posibles a cada transacción

descriptions = []
amounts = []

# Rangos (sensatos) de montos por categoría para entrenar con data que tenga sentido
rangos_montos = {
    'Alimentacion': (50, 400),
    'Transporte': (10, 80),
    'Salud': (40, 500),
    'Vivienda': (200, 1200),
    'Educacion': (100, 800),
    'Ocio': (20, 150),
    'Servicios': (30, 200),
    'Deudas': (150, 1000),
    'Ahorros': (100, 1500)
}

for cat in selected_categories: # recorre las 5000 opciones generadas--> categoría:descripción
    desc = np.random.choice(conceptos_categoria[cat]) # Toma una descripción aleatoria
    min_m, max_m = rangos_montos[cat] # Toma sus limites sensatos asociados
    monto = round(np.random.uniform(min_m, max_m), 2) # Genera un decimal al azar entre dichos limites con 2 decimales.
    descriptions.append(desc) # Guarda descripción + monto generado en sus respectivas listas para luego unirlas y armar la tabla final de transacciones.
    amounts.append(monto)

df_transacciones = pd.DataFrame({
    'id': range(1, n_transacciones + 1),
    'usuario_id': usuario_ids,
    'descripcion': descriptions,
    'valor': amounts,
    'categoria': selected_categories
})

print("Transacciones simuladas:", df_transacciones.shape)
print(df_transacciones['categoria'].value_counts())

Transacciones simuladas: (5000, 5)
categoria
Alimentacion    578
Servicios       569
Ahorros         563
Ocio            561
Deudas          553
Salud           549
Educacion       549
Transporte      541
Vivienda        537
Name: count, dtype: int64


## 4. Exportación de datasets "semilla"

Se crea el directorio data/ y se exportan los df en formato CSV para análisis y entrenamiento, y en formato JSON para servir de semilla en el backend.

In [4]:
os.makedirs('data', exist_ok=True)

# Exportar datos de usuarios
df_usuarios.to_csv('data/usuarios.csv', index=False)
df_usuarios.to_json('data/usuarios.json', orient='records', force_ascii=False, indent=2)

# Exportar datos de transacciones
df_transacciones.to_csv('data/transacciones.csv', index=False)
df_transacciones.to_json('data/transacciones.json', orient='records', force_ascii=False, indent=2)

print("¡Datasets generados y exportados exitosamente en la carpeta data/!")

¡Datasets generados y exportados exitosamente en la carpeta data/!
